# Notebook 5 — Customer Risk Scoring

**What this notebook does:** Scores every customer in the cleaned dataset with a
predicted churn probability using the best model, segments customers into
High/Medium/Low risk, and exports a Power BI-ready risk file.

**Input:** `models/best_model.pkl`, `data/processed/cleaned_churn.csv`, `models/scaler.pkl`, `models/feature_names.pkl`

**Output:**
- `data/powerbi/customer_risk_scores.csv`
- Charts saved to `outputs/charts/risk/`

**Estimated run time:** ~15 seconds

In [1]:
# Imports and shared project utilities
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import set_style, save_chart, save_dataframe, load_csv_safely, load_joblib_safely, annotate_vertical_bars, project_path

set_style()

In [2]:
# Load the best model, cleaned data, scaler, and feature name list
required_notebook_1 = "01_data_cleaning_eda.ipynb"
required_notebook_2 = "02_preprocessing.ipynb"
required_notebook_3 = "03_model_training.ipynb"

best_model = load_joblib_safely(project_path("models", "best_model.pkl"), required_notebook_3)
df_original = load_csv_safely(project_path("data", "processed", "cleaned_churn.csv"), required_notebook_1)
scaler = load_joblib_safely(project_path("models", "scaler.pkl"), required_notebook_2)
feature_names = load_joblib_safely(project_path("models", "feature_names.pkl"), required_notebook_2)

print("Customers to score:", df_original.shape[0])

Customers to score: 7032


## 1. Rebuild the same features used to train the model (no SMOTE — this is the full population, not a training set)

In [3]:
# Apply the identical feature engineering used in notebook 2
df_features = df_original.copy()

tenure_bins = [0, 12, 24, 36, 48, 60, 72]
tenure_labels = ["0-12 months", "13-24 months", "25-36 months",
                 "37-48 months", "49-60 months", "61-72 months"]
df_features["tenure_group"] = pd.cut(df_features["tenure"], bins=tenure_bins, labels=tenure_labels, include_lowest=True)

df_features["avg_monthly_spend"] = df_features["TotalCharges"] / (df_features["tenure"] + 1)
df_features["is_long_term_contract"] = df_features["Contract"].isin(["One year", "Two year"]).astype(int)
df_features["high_value_customer"] = (df_features["MonthlyCharges"] > df_features["MonthlyCharges"].median()).astype(int)

service_columns = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                   "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df_features["num_services"] = df_features[service_columns].apply(lambda col: (col == "Yes").astype(int)).sum(axis=1)

In [4]:
# Apply the identical encoding used in notebook 2
binary_columns = ["Partner", "Dependents", "PhoneService", "PaperlessBilling",
                   "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                   "TechSupport", "StreamingTV", "StreamingMovies", "SeniorCitizen"]
for col in binary_columns:
    df_features[col] = (df_features[col] == "Yes").astype(int)

onehot_columns = ["gender", "MultipleLines", "InternetService", "Contract", "PaymentMethod"]
df_features = pd.get_dummies(df_features, columns=onehot_columns, drop_first=True)

In [5]:
# Build the feature matrix and align its columns exactly with what the model was trained on
X_full = df_features.drop(columns=["customerID", "tenure_group", "Churn"])
X_full = X_full.astype({col: int for col in X_full.select_dtypes(include="bool").columns})
X_full = X_full.reindex(columns=feature_names, fill_value=0)

# Apply the saved scaler (fit on the training set in notebook 2) — do NOT refit here.
# Keep the result as a DataFrame with the original column names so scikit-learn doesn't
# warn about missing feature names when predicting.
X_full_scaled = pd.DataFrame(scaler.transform(X_full), columns=feature_names)

print("Feature matrix ready for scoring:", X_full_scaled.shape)

Feature matrix ready for scoring: (7032, 28)


## 2. Predict churn probability and assign risk segments

In [6]:
# Predict churn probability for every customer in the full dataset
churn_probability = best_model.predict_proba(X_full_scaled)[:, 1]

df_original["churn_probability"] = churn_probability
df_original["prediction"] = (df_original["churn_probability"] >= 0.50).astype(int)

risk_conditions = [
    df_original["churn_probability"] >= 0.70,
    df_original["churn_probability"] >= 0.40,
]
risk_choices = ["High Risk", "Medium Risk"]
df_original["risk_segment"] = np.select(risk_conditions, risk_choices, default="Low Risk")

print("Risk scoring complete.")
df_original[["customerID", "churn_probability", "risk_segment", "prediction"]].head()

Risk scoring complete.


,customerID,churn_probability,risk_segment,prediction
0,7590-VHVEG,0.748607,High Risk,1
1,5575-GNVDE,0.043123,Low Risk,0
2,3668-QPYBK,0.285420,Low Risk,0
3,7795-CFOCW,0.058606,Low Risk,0
4,9237-HQITU,0.851765,High Risk,1


In [7]:
# Distribution of customers across risk segments (count + %)
segment_counts = df_original["risk_segment"].value_counts()
segment_pct = df_original["risk_segment"].value_counts(normalize=True).mul(100).round(1)

print("Risk segment distribution (count / %):")
for segment in ["High Risk", "Medium Risk", "Low Risk"]:
    print(f"  {segment}: {segment_counts.get(segment, 0)} customers ({segment_pct.get(segment, 0)}%)")

Risk segment distribution (count / %):
  High Risk: 979 customers (13.9%)
  Medium Risk: 1581 customers (22.5%)
  Low Risk: 4472 customers (63.6%)


In [8]:
# Average churn probability, MonthlyCharges, and tenure per segment
segment_summary = df_original.groupby("risk_segment").agg(
    avg_churn_probability=("churn_probability", "mean"),
    avg_monthly_charges=("MonthlyCharges", "mean"),
    avg_tenure=("tenure", "mean"),
).reindex(["High Risk", "Medium Risk", "Low Risk"]).round(2)

print("Segment summary:")
print(segment_summary)

Segment summary:
              avg_churn_probability  avg_monthly_charges  avg_tenure
risk_segment                                                        
High Risk                      0.81                76.80        6.70
Medium Risk                    0.55                72.52       20.27
Low Risk                       0.12                59.44       42.35


## 3. Risk scoring charts

### Chart 1 — Risk segment distribution

In [9]:
# Pie chart of risk segment share, with % labels
fig, ax = plt.subplots(figsize=(12, 6))
colors = {"High Risk": "#C44E52", "Medium Risk": "#DD8452", "Low Risk": "#55A868"}
ordered_segments = ["High Risk", "Medium Risk", "Low Risk"]
values = [segment_counts.get(s, 0) for s in ordered_segments]
ax.pie(values, labels=ordered_segments, autopct="%1.1f%%", colors=[colors[s] for s in ordered_segments], startangle=90)
ax.set_title("Risk Segment Distribution", fontsize=14, fontweight="bold")

save_chart(fig, project_path("outputs", "charts", "risk", "risk_segment_distribution.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/risk/risk_segment_distribution.png (59.5KB)


### Chart 2 — Monthly Charges by risk segment

In [10]:
# Bar chart of average MonthlyCharges per segment, annotated with £ values
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=segment_summary.index, y=segment_summary["avg_monthly_charges"],
            hue=segment_summary.index, palette=[colors[s] for s in segment_summary.index],
            legend=False, ax=ax)
ax.set_title("Average Monthly Charges by Risk Segment", fontsize=14, fontweight="bold")
ax.set_xlabel("Risk Segment")
ax.set_ylabel("Average Monthly Charges (£)")
annotate_vertical_bars(ax, fmt="£{:.2f}")

save_chart(fig, project_path("outputs", "charts", "risk", "monthly_charges_by_risk.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/risk/monthly_charges_by_risk.png (57.7KB)


### Chart 3 — Tenure by risk segment

In [11]:
# Bar chart of average tenure per segment, annotated with month values
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=segment_summary.index, y=segment_summary["avg_tenure"],
            hue=segment_summary.index, palette=[colors[s] for s in segment_summary.index],
            legend=False, ax=ax)
ax.set_title("Average Tenure by Risk Segment", fontsize=14, fontweight="bold")
ax.set_xlabel("Risk Segment")
ax.set_ylabel("Average Tenure (months)")
annotate_vertical_bars(ax, fmt="{:.1f} months")

save_chart(fig, project_path("outputs", "charts", "risk", "tenure_by_risk.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/risk/tenure_by_risk.png (53.5KB)


### Chart 4 — Churn probability distribution

In [12]:
# Histogram of churn probability across all customers, with segment boundary lines
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df_original["churn_probability"], bins=40, color="#4C72B0", alpha=0.8)
ax.axvline(0.40, color="#DD8452", linestyle="--", linewidth=2, label="Medium Risk boundary (0.40)")
ax.axvline(0.70, color="#C44E52", linestyle="--", linewidth=2, label="High Risk boundary (0.70)")
ax.set_title("Churn Probability Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Churn Probability")
ax.set_ylabel("Number of Customers")
ax.legend()

save_chart(fig, project_path("outputs", "charts", "risk", "churn_probability_distribution.png"))

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/outputs/charts/risk/churn_probability_distribution.png (64.4KB)


### Chart 5 — Top 20 highest-risk customers

In [13]:
# Print the top 20 highest-risk customers as a formatted table
top_20_high_risk = df_original.sort_values("churn_probability", ascending=False).head(20)
top_20_display = top_20_high_risk[
    ["customerID", "Contract", "tenure", "MonthlyCharges", "InternetService", "churn_probability", "risk_segment"]
].reset_index(drop=True)

print(top_20_display.to_string(index=False))

customerID       Contract  tenure  MonthlyCharges InternetService  churn_probability risk_segment
4910-GMJOT Month-to-month       1           94.60     Fiber optic           0.967890    High Risk
5419-JPRRN Month-to-month       1          101.45     Fiber optic           0.966554    High Risk
8149-RSOUN Month-to-month       1           93.85     Fiber optic           0.964675    High Risk
0295-PPHDO Month-to-month       1           95.45     Fiber optic           0.963335    High Risk
7216-EWTRS Month-to-month       1          100.80     Fiber optic           0.961394    High Risk
3988-RQIXO Month-to-month       1           91.30     Fiber optic           0.960199    High Risk
3158-MOERK Month-to-month       2           96.00     Fiber optic           0.957757    High Risk
5797-APWZC Month-to-month       1           90.60     Fiber optic           0.957172    High Risk
7294-TMAOP Month-to-month       1           90.55     Fiber optic           0.956901    High Risk
5192-EBGOV Month-to-

## 4. Save the Power BI risk export

In [14]:
# Save the customer-level risk scores file for Power BI
powerbi_columns = ["customerID", "gender", "SeniorCitizen", "Partner", "tenure",
                    "Contract", "MonthlyCharges", "TotalCharges", "InternetService",
                    "TechSupport", "Churn", "churn_probability", "risk_segment"]

risk_scores_export = df_original[powerbi_columns]
risk_scores_path = project_path("data", "powerbi", "customer_risk_scores.csv")
save_dataframe(risk_scores_export, risk_scores_path)

print(f"\nRow count: {risk_scores_export.shape[0]}")
print(f"Columns: {list(risk_scores_export.columns)}")

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/powerbi/customer_risk_scores.csv (667.0KB) — 7032 rows, 13 columns

Row count: 7032
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'tenure', 'Contract', 'MonthlyCharges', 'TotalCharges', 'InternetService', 'TechSupport', 'Churn', 'churn_probability', 'risk_segment']
